# 50 -- enumeration probe, step 4: the one honest read

Touches the 8-video held-out set EXACTLY ONCE. Refits each task's winning (layer, kind, C)
from `03`'s CV sweep on the FULL fit pool (all 122 videos, no folds held back this time),
then reads it against `02`'s final-read dump.

**⚠️ Effective n is 8 VIDEOS, not the row count** (RULES §13 -- `frame.metrics` clusters
bootstrap CIs by video for exactly this reason). Every headline below is a
`frame.metrics._hier_bootstrap` video->question two-level bootstrap, reused rather than
reimplemented, not a flat row-mean.

Three things get reported, each against the SAME 8 videos:
1. **Probe vs token-head**, per task -- does the hidden-state probe beat the model's own
   greedy-decoding confidence (rung 33/34's thesis check, `probe > token_head`)?
2. **Aggregation comparison**: for `n_classes` number questions that share an EXACT frame
   (same video + same `timestamp_start`) with an fo_class question in the final-read set,
   does `len(fo_class-probe's predicted set)` recover what the counting probe predicts for
   the SAME underlying scene? Only 20 such exact-frame pairs exist in this 8-video set --
   reported as its own small subset, never folded into the two main headlines.

In [ ]:
# --- parameters (RAW LITERALS ONLY -- papermill injects a new cell right after THIS one) --
HIDDEN_DIR = "/workspace/repo_yyy/experiments/56-enumeration-probe/runs/50_hidden_v1"
TAG = "full"
MODEL_PATH = "/workspace/repo_yyy/experiments/49-flip-equivariance/runs/49_flip_pair_v1/merged/checkpoint-4848"
DATA_ROOT = "/workspace/orena-data"
FRAMES_CACHE = "/workspace/frames_cache"
PCA_N_COMPONENTS = 128
N_BOOT = 1000
SEED = 42

In [ ]:
# --- bootstrap -----------------------------------------------------------------
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

EXP = Path.cwd()
REPO = EXP
while REPO != REPO.parent and not (REPO / "src").is_dir():
    REPO = REPO.parent
if EXP.name != "56-enumeration-probe":
    EXP = REPO / "experiments" / "56-enumeration-probe"

if str(EXP / "_tools") not in sys.path:
    sys.path.insert(0, str(EXP / "_tools"))
for p in (REPO / "src", REPO / "vendor" / "orena-focus" / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import ordinal_probe as OP
import multilabel_probe as MP
import token_head as TH
import resolve_frames
from frame.config import BaselineConfig
from frame.metrics import _hier_bootstrap

HIDDEN_DIR = Path(HIDDEN_DIR)
print("repo:", REPO, "| exp:", EXP, "| hidden_dir:", HIDDEN_DIR)

In [ ]:
# --- derived --------------------------------------------------------------------
DATA_ROOT = next(
    (Path(c) for c in (DATA_ROOT, str(REPO / "external_data" / "orena-data"))
     if (Path(c) / "heico" / "data" / "frame" / "test.parquet").exists()
     or (Path(c) / "heico" / "test.parquet").exists()),
    None)
assert DATA_ROOT is not None, "no frame test.parquet found (checked nested and flat layouts)"

cfg = BaselineConfig(data_root=DATA_ROOT)
cfg.frames_cache = Path(FRAMES_CACHE)
assert cfg.frames_cache.is_dir(), f"frames_cache not found: {cfg.frames_cache}"


def _vkey(row):
    return (row["dataset"], row["video"])


def _bootstrap_ci(correct: np.ndarray, vkeys) -> dict:
    sub = pd.DataFrame({"_correct": correct.astype(float), "_vkey": list(vkeys)})
    mean, lo, hi = _hier_bootstrap(sub, n_boot=N_BOOT, rng=np.random.default_rng(SEED))
    return {"mean": mean, "ci_low": lo, "ci_high": hi, "n_rows": len(sub),
            "n_videos": sub["_vkey"].nunique()}

In [ ]:
# --- 1. load dumps + manifests + the winning config from 03 ------------------------
feats_fit = np.load(HIDDEN_DIR / f"feats_fit_{TAG}.npy")
meta_fit = pd.read_csv(HIDDEN_DIR / f"meta_fit_{TAG}.csv")
layers_fit = json.loads((HIDDEN_DIR / f"layers_fit_{TAG}.json").read_text())

feats_final = np.load(HIDDEN_DIR / f"feats_final_read_{TAG}.npy")
meta_final = pd.read_csv(HIDDEN_DIR / f"meta_final_read_{TAG}.csv")
layers_final = json.loads((HIDDEN_DIR / f"layers_final_read_{TAG}.json").read_text())
assert layers_fit == layers_final, "fit/final-read dumps must share the same kept-layer list"

best = json.loads((EXP / "RESULTS_best_config.json").read_text())
print(json.dumps(best, indent=2))

num_fit_m = pd.read_csv(EXP / "RESULTS_fit_number_v1.csv")
fo_fit_m = pd.read_csv(EXP / "RESULTS_fit_foclass_v1.csv")
final_m = pd.read_csv(EXP / "RESULTS_final_read_v1.csv")
num_final_m = final_m[final_m.answer_format == "number"].copy()
fo_final_m = final_m[final_m.answer_format == "fo_class"].copy()


def _positions(manifest: pd.DataFrame, meta: pd.DataFrame) -> np.ndarray:
    pos = {q: i for i, q in enumerate(meta["qID"])}
    missing = [q for q in manifest["qID"] if q not in pos]
    assert not missing, f"{len(missing)} qIDs missing from the dump (e.g. {missing[:5]})"
    return np.array([pos[q] for q in manifest["qID"]])


num_fit_pos = _positions(num_fit_m, meta_fit)
fo_fit_pos = _positions(fo_fit_m, meta_fit)
num_final_pos = _positions(num_final_m, meta_final)
fo_final_pos = _positions(fo_final_m, meta_final)

In [ ]:
# --- 2. counting: refit on the FULL fit pool at the winning layer/kind/C, read once ---
cnt_cfg = best["counting"]
li = layers_fit.index(cnt_cfg["layer"])
X_train = feats_fit[num_fit_pos][:, li, :].astype("float32")
y_train = num_fit_m["answer"].astype(int).to_numpy()
X_test = feats_final[num_final_pos][:, li, :].astype("float32")
y_test = num_final_m["answer"].astype(int).to_numpy()

fit_result, _, proba_test = OP.fit(X_train, y_train, X_test, kind=cnt_cfg["kind"],
                                   layer=cnt_cfg["layer"], C=cnt_cfg["C"],
                                   n_components=PCA_N_COMPONENTS, seed=SEED)
num_final_m["probe_pred"] = OP.predict(proba_test, fit_result.classes)
num_final_m["probe_correct"] = (num_final_m["probe_pred"] == y_test)

probe_num_ci = _bootstrap_ci(num_final_m["probe_correct"].to_numpy(),
                             num_final_m.apply(_vkey, axis=1))
print("counting probe:", probe_num_ci)

In [ ]:
# --- 3. fo_class: refit on the FULL fit pool, vocab = UNION of fit + final-read labels --
# multilabel_probe.py's own contract: a label seen only at read time must still be a
# vocab column, or it can never be predicted and silently counts as a probe miss.
fo_cfg = best["fo_class"]
vocab = MP.build_vocab(pd.concat([fo_fit_m["answer"], fo_final_m["answer"]]))
if vocab != fo_cfg["vocab"]:
    print(f"vocab grew at read time: fit-only had {fo_cfg['vocab']}, union is {vocab}")

lj = layers_fit.index(fo_cfg["layer"])
X_train = feats_fit[fo_fit_pos][:, lj, :].astype("float32")
X_test = feats_final[fo_final_pos][:, lj, :].astype("float32")

ml_fit, _, proba_test = MP.fit(X_train, fo_fit_m["answer"].to_numpy(), X_test, vocab=vocab,
                               layer=fo_cfg["layer"], C=fo_cfg["C"],
                               n_components=PCA_N_COMPONENTS, seed=SEED)
pred_sets = MP.predict_sets(proba_test, vocab)
fo_final_m["probe_pred_set"] = [",".join(sorted(s)) if s else "none" for s in pred_sets]
fo_final_m["probe_correct"] = MP.exact_set_match(pred_sets, fo_final_m["answer"].to_numpy())

probe_fo_ci = _bootstrap_ci(fo_final_m["probe_correct"].to_numpy(),
                            fo_final_m.apply(_vkey, axis=1))
print("fo_class probe:", probe_fo_ci)

## Token-head baselines (GPU)

Reuses rung 33's exact `P(value)` mechanism for `number`, and the simpler joint-log-prob
mechanism `token_head.py` documents for `fo_class` (class names are multi-token; rung 33's
per-value single-token trick does not port -- see that module's docstring for the
consequence: this is a single scalar confidence, not a per-class marginal).

In [ ]:
# --- 4. token-head: resolve frames, run both GPU dumps on the SAME final-read rows ----
num_final_resolved = resolve_frames.resolve_image_paths(num_final_m, cfg)
fo_final_resolved = resolve_frames.resolve_image_paths(fo_final_m, cfg)

th_num = TH.dump_number_distribution(MODEL_PATH, num_final_resolved)
num_final_m = num_final_m.merge(th_num, on="qID")
num_final_m["token_head_correct"] = (num_final_m["token_head_pred"] == y_test)
token_num_ci = _bootstrap_ci(num_final_m["token_head_correct"].to_numpy(),
                             num_final_m.apply(_vkey, axis=1))
print("counting token-head:", token_num_ci)

In [ ]:
# --- 5. token-head: fo_class -- parse the model's own free-text answer into a label set --
th_fo = TH.dump_foclass_confidence(MODEL_PATH, fo_final_resolved)
fo_final_m = fo_final_m.merge(th_fo, on="qID")
th_pred_sets = [MP.parse_labels(a) for a in fo_final_m["token_head_answer"]]
fo_final_m["token_head_correct"] = MP.exact_set_match(th_pred_sets, fo_final_m["answer"].to_numpy())
token_fo_ci = _bootstrap_ci(fo_final_m["token_head_correct"].to_numpy(),
                            fo_final_m.apply(_vkey, axis=1))
print("fo_class token-head:", token_fo_ci)

In [ ]:
# --- 6. headline: probe vs token-head, per task -------------------------------------
headline = {
    "counting": {"probe": probe_num_ci, "token_head": token_num_ci,
                 "probe_beats_token_head": probe_num_ci["mean"] > token_num_ci["mean"]},
    "fo_class": {"probe": probe_fo_ci, "token_head": token_fo_ci,
                 "probe_beats_token_head": probe_fo_ci["mean"] > token_fo_ci["mean"]},
}
print(json.dumps(headline, indent=2))

In [ ]:
# --- 7. aggregation comparison: exact-frame co-occurrence subset only ---------------
n_classes_rows = num_final_m[num_final_m["_subtmpl"] == "n_classes"].copy()
paired = n_classes_rows.merge(
    fo_final_m, on=["_video_key", "timestamp_start"], suffixes=("_num", "_fo"),
)
print(f"{len(n_classes_rows)} n_classes rows, {len(paired)} with an exact-frame fo_class match")

if len(paired):
    # NOTE: `probe_pred` / `probe_pred_set` only exist on one side each (num / fo) --
    # pandas only appends a suffix to columns that COLLIDE by name, so these two stay
    # bare after the merge (unlike `answer_num`/`answer_fo`, which do collide).
    paired["agg_count"] = paired["probe_pred_set"].map(
        lambda s: 0 if s == "none" else len(s.split(",")))
    paired["gold_count"] = paired["answer_num"].astype(int)
    paired["probe_count_correct"] = paired["probe_pred"] == paired["gold_count"]
    paired["agg_count_correct"] = paired["agg_count"] == paired["gold_count"]
    paired["probe_agrees_with_agg"] = paired["probe_pred"] == paired["agg_count"]

    aggregation_comparison = {
        "n_pairs": int(len(paired)),
        "counting_probe_acc_on_pairs": float(paired["probe_count_correct"].mean()),
        "aggregated_fo_class_acc_on_pairs": float(paired["agg_count_correct"].mean()),
        "probe_agg_agreement_rate": float(paired["probe_agrees_with_agg"].mean()),
    }
else:
    aggregation_comparison = {"n_pairs": 0, "note": "no exact-frame co-occurrence in this pool"}
print(json.dumps(aggregation_comparison, indent=2))

In [ ]:
# --- 8. persist -----------------------------------------------------------------
result = {
    "headline": headline,
    "aggregation_comparison": aggregation_comparison,
    "best_config": best,
    "n_boot": N_BOOT,
    "seed": SEED,
}
(EXP / "RESULTS_final_read.json").write_text(json.dumps(result, indent=2))
num_final_m.to_csv(EXP / "RESULTS_final_read_number_detail.csv", index=False)
fo_final_m.to_csv(EXP / "RESULTS_final_read_foclass_detail.csv", index=False)
print("wrote RESULTS_final_read.json + per-task detail CSVs")